# Field - Rust

All 14 Rust examples from [docs/field.md](https://platob.github.io/yggdryl/field/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::{DataType, Field};

let field = Field::new("price", DataType::from_str("decimal(18, 6)")?, false);

assert_eq!(field.name(), "price");
assert_eq!(field.dtype(), &DataType::decimal(18, 6)?);
assert!(!field.is_nullable());
assert!(field.is_metadata_empty());

// The canonical text round-trips, and shorthand parses into the same value.
assert_eq!(Field::from_str(&field.to_string())?, field);
assert_eq!(Field::from_str("price decimal(18, 6) NOT NULL")?, field);

## A non-null struct field is the schema

In [ ]:
use yggdryl::{DataType, Field};

let schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?
.required_field("trade");

schema.validate_struct_root()?;
assert_eq!(schema.field_len(), 2);
assert_eq!(schema.index_of("symbol"), Some(1));
assert_eq!(schema.get_field_by_name("id").map(Field::name), Some("id"));

// A nullable root is not a schema: a whole row cannot be logically absent.
assert!(schema.with_nullable(true).validate_struct_root().is_err());

### Item access reaches a child, never metadata

In [ ]:
use yggdryl::{DataType, Field};

let mut order = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::from_fields([DataType::Float64.required_field("price")])?
        .required_field("line"),
])?
.required_field("order");
order.insert_metadata("owner", "trading")?;

// A child by name, by position, and two levels down.
assert_eq!(order["id"].dtype(), &DataType::Int64);
assert_eq!(order[1].name(), "line");
assert_eq!(order["line"]["price"].dtype(), &DataType::Float64);

// An unknown name appends; a position replaces.
order.set_field_by_name("venue", DataType::Utf8.nullable_field("venue"))?;
assert_eq!(order.field_len(), 3);
order.set_field(0, DataType::Utf8.required_field("id"))?;
assert_eq!(order["id"].dtype(), &DataType::Utf8);
assert_eq!(order.remove_field_by_name("venue")?.name(), "venue");

// Metadata keeps its own named surface.
assert_eq!(order.get_metadata("owner"), Some("trading"));
assert!(order.get_field_by_name("owner").is_none());

## Metadata is a mapping

In [ ]:
use yggdryl::{DataType, Field};

let mut field = Field::from_parts("price", DataType::Float64, false, [("venue", "XPAR")])?;
field.insert_metadata("currency", "EUR")?;
field.update_metadata([("source", "exchange")])?;

assert_eq!(field.metadata_len(), 3);
assert_eq!(field.get_metadata("venue"), Some("XPAR"));
assert!(field.has_metadata("currency"));
assert_eq!(
    field.metadata_iter().collect::<Vec<_>>(),
    [("currency", "EUR"), ("source", "exchange"), ("venue", "XPAR")]
);
assert_eq!(field.remove_metadata("venue").as_deref(), Some("XPAR"));

## Reserved keys and protocol properties

In [ ]:
use yggdryl::{DataType, Field, MimeType, Scheme};

let mut field = Field::new("payload", DataType::Binary, false);

field.set_parquet_field_id(17);
field.set_init(false);
field.set_content_type("application/json; charset=utf-8")?;
field.set_property(&Scheme::POSTGRES, "type", "jsonb")?;

assert_eq!(field.parquet_field_id()?, Some(17));
assert_eq!(field.get_metadata("PARQUET:field_id"), Some("17"));
assert!(!field.is_init()?);
assert_eq!(field.get_metadata("field:init"), Some("false"));

// An http: property answers to either scheme and to a raw key lookup.
assert_eq!(field.mime_type()?, MimeType::JSON);
assert_eq!(
    field.get_property(&Scheme::HTTPS, "Content-Type"),
    field.content_type()
);
assert_eq!(field.get_metadata("http:content-type"), field.content_type());
assert_eq!(
    field.property_iter(&Scheme::POSTGRES).collect::<Vec<_>>(),
    [("type", "jsonb")]
);

## One protocol at a time

In [ ]:
use yggdryl::{DataType, Field, Scheme};

let mut field = Field::new("price", DataType::Int64, false);

field.iceberg_mut().insert("doc", "closing price")?;
field.iceberg_mut().update([("schema-id", "3"), ("field-id", "7")])?;
field.postgres_mut().insert("type", "numeric")?;

assert_eq!(field.iceberg().get("doc"), Some("closing price"));
assert_eq!(field.iceberg().key("doc"), "iceberg:doc");
assert_eq!(field.iceberg().len(), 3);
assert!(field.mysql().is_empty());

// It is a view of the one metadata map, not a copy of part of it.
assert_eq!(field.get_metadata("iceberg:doc"), Some("closing price"));
assert_eq!(field.metadata_len(), 4);

// A protocol-scoped replacement leaves every other protocol alone.
field.iceberg_mut().set([("doc", "close")])?;
assert_eq!(field.iceberg().iter().collect::<Vec<_>>(), [("doc", "close")]);
assert_eq!(field.postgres().get("type"), Some("numeric"));

// The protocol can also come from a value rather than from the code.
assert_eq!(field.protocol(&Scheme::POSTGRES).get("type"), Some("numeric"));

## A field can be a partition column

In [ ]:
use yggdryl::DataType;

let schema = DataType::from_fields([
    DataType::Int32.required_field("year"),
    DataType::Utf8.required_field("venue"),
    DataType::Int64.required_field("price"),
])?
.required_field("row")
.with_partition_fields(&["year", "venue"])?;

assert!(schema.has_partition_fields());
assert_eq!(schema.partition_field_names().collect::<Vec<_>>(), ["year", "venue"]);
assert!(schema.get_field_by_name("year").expect("the column").is_partition());

// The two halves of the layout: what a path spells, and what a leaf stores.
assert_eq!(schema.without_partition_fields()?.field_len(), 1);
assert_eq!(schema.only_partition_fields()?.field_len(), 2);

// The mark is reserved metadata, so it round-trips like any other.
assert_eq!(
    schema.get_field_by_name("year").expect("the column").get_metadata("field:partition"),
    Some("true")
);

## Typed field aliases

In [ ]:
use yggdryl::field::{Int64Field, TimestampField, Utf8Field, integer};
use yggdryl::{DataType, Field, TimeUnit};

let id = Int64Field::new("id", false);
let symbol = Utf8Field::from_parts("symbol", true, [("source", "feed")])?;
let at = TimestampField::try_new("at", DataType::Timestamp(TimeUnit::Microsecond, None), false)?;

// A typed field derefs to the field it wraps.
assert_eq!(id.name(), "id");
assert_eq!(symbol.get_metadata("source"), Some("feed"));
assert_eq!(at.dtype().to_string(), "timestamp(us)");

// The marker is checked, never assumed.
assert!(
    Field::new("id", DataType::Utf8, false)
        .try_into_typed::<integer::Int64>()
        .is_err()
);
assert_eq!(id.into_field().dtype(), &DataType::Int64);

## Row values are validated against the root

In [ ]:
use yggdryl::{DataType, Field, Scalar};

let schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Float32.nullable_field("price"),
])?
.required_field("trade");

// A row is one ordered sequence with one value per struct child.
let row = Scalar::from_sequence([Scalar::from(7u64), Scalar::from(0.1f64)]);
schema.validate_value(&row)?;

// Canonicalizing narrows every value into the representation the root declares.
let canonical = schema.canonicalize_value(row)?;
assert_eq!(canonical.get(0), Some(&Scalar::I64(7)));
assert_eq!(
    canonical.get(1).and_then(Scalar::as_f64),
    Some(f64::from(0.1f32))
);

// A value that does not fit names the path walked to reach it.
let wrong = Scalar::from_sequence([Scalar::from("seven"), Scalar::Null]);
let message = schema.validate_value(&wrong).unwrap_err().to_string();
assert!(message.contains("$.trade.id"), "{message}");

## Serializing a schema

In [ ]:
use yggdryl::{DataType, Field};
use yggdryl::generic::Scalar;

let field = Field::from_parts("price", DataType::Float64, false, [("venue", "XPAR")])?;

// One structural model, three formats over it.
assert_eq!(Field::from_value(field.clone().into_value())?, field);
assert_eq!(Field::from_json(&field.clone().into_json()?)?, field);
assert_eq!(Field::from_yaml(&field.clone().into_yaml()?)?, field);
assert_eq!(Field::from_toml(&field.clone().into_toml()?)?, field);

// The mapping is the shared `Scalar`, so it drops into any document.
let shape = field.into_value();
assert_eq!(shape.get_key_str("name").and_then(Scalar::as_utf8), Some("price"));
// Unset optional attributes are absent rather than null.
assert!(shape.get_key_str("dictionary_id").is_none());

## A readable rendering

In [ ]:
use yggdryl::{DataType, Field};

let order = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::from_fields([DataType::Float64.required_field("price")])?
        .nullable_field("line"),
])?
.required_field("order");

// Compact still round-trips.
assert_eq!(Field::from_str(&order.to_string())?, order);

// Readable is the alternate, or the named adapter - one implementation.
assert_eq!(format!("{order:#}"), order.pretty().to_string());
assert_eq!(
    format!("{order:#}"),
    concat!(
        "order: struct[2], required\n",
        "  id: int64, required\n",
        "  line: struct[1], nullable\n",
        "    price: float64, required",
    ),
);

## Comparing two fields

In [ ]:
use yggdryl::{DataType, Field};

let left = Field::from_parts("price", DataType::Float64, false, [("venue", "XPAR")])?;
let right = Field::from_parts("price", DataType::Float64, true, [("venue", "XNAS")])?;

assert!(!left.equals(&right, true));
assert_eq!(
    left.show_diffs(&right, true, false).collect::<Vec<_>>(),
    [
        "≠ $.nullable: false → true",
        "≠ $.metadata[\"venue\"]: \"XPAR\" → \"XNAS\"",
    ]
);
assert_eq!(left.show_diff(&left, true, true), "✓ equal");
assert_eq!(left.show_diff(&left, true, false), "");

## Casting Arrow data through a field

In [ ]:
use std::sync::Arc;

use arrow_array::{Array, ArrayRef, Int64Array, StringArray};
use yggdryl::field::Int64Field;
use yggdryl::{ArrowCast, DataType, Field};

let text: ArrayRef = Arc::new(StringArray::from(vec!["1", "2"]));

// Any field answers with an ArrayRef, because any field could be any datatype.
let field = Field::new("id", DataType::Int64, false);
let cast = field.cast_arrow_array(Arc::clone(&text), false)?;
assert_eq!(cast.dtype(), &arrow_schema::DataType::Int64);

// A typed field already knows its variant, so it answers with the array itself.
let typed = Int64Field::new("id", false);
let ids: Int64Array = typed.cast_arrow_array(text, false)?;
assert_eq!(ids.values(), &[1, 2]);

// safe nulls a failed conversion; a non-null field then defaults it.
let broken: ArrayRef = Arc::new(StringArray::from(vec!["1", "not a number"]));
assert!(typed.cast_arrow_array(Arc::clone(&broken), false).is_err());
let repaired: Int64Array = typed.cast_arrow_array(broken, true)?;
assert_eq!(repaired.values(), &[1, 0]);
assert_eq!(repaired.null_count(), 0);

In [ ]:
use std::sync::Arc;

use arrow_array::{Int32Array, RecordBatch, StringArray};
use arrow_schema::{DataType as ArrowDataType, Field as ArrowField, Schema};
use yggdryl::{ArrowCast, DataType, Field};

let schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?
.required_field("trade");

let source = RecordBatch::try_new(
    Arc::new(Schema::new(vec![
        ArrowField::new("symbol", ArrowDataType::Utf8, true),
        ArrowField::new("id", ArrowDataType::Int32, false),
    ])),
    vec![
        Arc::new(StringArray::from(vec!["ACME"])),
        Arc::new(Int32Array::from(vec![7])),
    ],
)?;

let batch = schema.cast_arrow_batch(source, false)?;
assert_eq!(batch.num_columns(), 2);
assert_eq!(batch.schema().field(0).name(), "id");
assert_eq!(batch.column(0).dtype(), &ArrowDataType::Int64);